# Simulated Annealing for Supply Chain Logistics Optimization

This notebook implements a Simulated Annealing metaheuristic for the supply chain logistics optimization problem.

The objective is to minimize total logistics cost while satisfying the relevant supply chain constraints, including product–plant compatibility, plant–port connectivity, plant capacity, and transportation feasibility.

The same optimization problem and processed data will be used for comparison with the MILP approach.

In [ ]:
import pandas as pd
import numpy as np
import random
import math
import time

import matplotlib.pyplot as plt

In [ ]:
data = pd.read_excel(
    "data/cleaned_supply_chain.xlsx",
    sheet_name=None
)

print("Data loaded successfully.")
print("Sheets:", list(data.keys()))

Data loaded successfully.
Sheets: ['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts']


In [ ]:
orders = data["OrderList"]
freight_rates = data["FreightRates"]
wh_costs = data["WhCosts"]
wh_capacities = data["WhCapacities"]
products_plant = data["ProductsPerPlant"]
vmi_customers = data["VmiCustomers"]
plant_ports = data["PlantPorts"]

print("DataFrames created successfully.")

DataFrames created successfully.


In [ ]:
print(orders[
    [
        "Order ID",
        "Origin Port",
        "Carrier",
        "Service Level",
        "Customer",
        "Product ID",
        "Plant Code",
        "Destination Port",
        "Unit quantity",
        "Weight"
    ]
].head())

       Order ID Origin Port Carrier Service Level   Customer  Product ID  \
0  1.447296e+09      PORT09   V44_3           CRF  V55555_53     1700106   
1  1.447158e+09      PORT09   V44_3           CRF  V55555_53     1700106   
2  1.447139e+09      PORT09   V44_3           CRF  V55555_53     1700106   
3  1.447364e+09      PORT09   V44_3           CRF  V55555_53     1700106   
4  1.447364e+09      PORT09   V44_3           CRF  V55555_53     1700106   

  Plant Code Destination Port  Unit quantity  Weight  
0    PLANT16           PORT09            808   14.30  
1    PLANT16           PORT09           3188   87.94  
2    PLANT16           PORT09           2331   61.20  
3    PLANT16           PORT09            847   16.16  
4    PLANT16           PORT09           2163   52.34  


In [ ]:
product_to_plants = (
    products_plant
    .groupby("Product ID")["Plant Code"]
    .apply(list)
    .to_dict()
)

print("Number of products:", len(product_to_plants))

# Show a few examples
for product, plants in list(product_to_plants.items())[:10]:
    print(product, "→", plants)

Number of products: 1540
1613321 → ['PLANT03']
1615677 → ['PLANT03']
1617714 → ['PLANT12']
1617826 → ['PLANT12']
1620510 → ['PLANT03']
1620552 → ['PLANT03']
1621717 → ['PLANT12']
1621719 → ['PLANT12', 'PLANT13']
1622685 → ['PLANT03']
1623991 → ['PLANT03']


In [ ]:
ordered_products = orders["Product ID"].unique()

ordered_product_to_plants = {
    product: product_to_plants.get(product, [])
    for product in ordered_products
}

print("Products appearing in orders:", len(ordered_products))

# Check whether every ordered product has at least one feasible plant
products_without_plants = [
    product
    for product, plants in ordered_product_to_plants.items()
    if len(plants) == 0
]

print("Products without feasible plants:", len(products_without_plants))

Products appearing in orders: 651
Products without feasible plants: 0


In [ ]:
#Plant → Feasible Ports

plant_to_ports = (
    plant_ports
    .groupby("Plant Code")["Port"]
    .apply(list)
    .to_dict()
)

print("Number of plants:", len(plant_to_ports))

for plant, ports in plant_to_ports.items():
    print(plant, "→", ports)

Number of plants: 19
PLANT01 → ['PORT01', 'PORT02']
PLANT02 → ['PORT03']
PLANT03 → ['PORT04']
PLANT04 → ['PORT05']
PLANT05 → ['PORT06']
PLANT06 → ['PORT06']
PLANT07 → ['PORT01', 'PORT02']
PLANT08 → ['PORT04']
PLANT09 → ['PORT04']
PLANT10 → ['PORT01', 'PORT02']
PLANT11 → ['PORT04']
PLANT12 → ['PORT04']
PLANT13 → ['PORT04']
PLANT14 → ['PORT07']
PLANT15 → ['PORT08']
PLANT16 → ['PORT09']
PLANT17 → ['PORT10']
PLANT18 → ['PORT11']
PLANT19 → ['PORT04']


In [ ]:
print(freight_rates.head(10))

  Carrier orig_port_cd dest_port_cd  minm_wgh_qty  max_wgh_qty svc_cd  \
0  V444_6       PORT08       PORT09         250.0       499.99    DTD   
1  V444_6       PORT08       PORT09          65.0        69.99    DTD   
2  V444_6       PORT08       PORT09          60.0        64.99    DTD   
3  V444_6       PORT08       PORT09          50.0        54.99    DTD   
4  V444_6       PORT08       PORT09          35.0        39.99    DTD   
5  V444_6       PORT08       PORT09         100.0       249.99    DTD   
6  V444_6       PORT08       PORT09         500.0      1999.99    DTD   
7  V444_6       PORT08       PORT09          55.0        59.99    DTD   
8  V444_6       PORT08       PORT09          25.0        29.99    DTD   
9  V444_6       PORT08       PORT09          30.0        34.99    DTD   

   minimum cost    rate mode_dsc  tpt_day_cnt Carrier type  
0       43.2272  0.7132   AIR               2  V88888888_0  
1       43.2272  0.7512   AIR               2  V88888888_0  
2       43.22

In [ ]:
print("Freight rate rows:", len(freight_rates))
print("Origin ports:", freight_rates["orig_port_cd"].nunique())
print("Destination ports:", freight_rates["dest_port_cd"].nunique())
print("Carriers:", freight_rates["Carrier"].nunique())
print("Service codes:", freight_rates["svc_cd"].nunique())

Freight rate rows: 1540
Origin ports: 10
Destination ports: 1
Carriers: 9
Service codes: 2


In [ ]:
# Phase 3: Create basic lookup dictionaries

# Product → feasible plants
product_to_plants = (
    products_plant
    .groupby("Product ID")["Plant Code"]
    .apply(list)
    .to_dict()
)

# Plant → connected ports
plant_to_ports = (
    plant_ports
    .groupby("Plant Code")["Port"]
    .apply(list)
    .to_dict()
)

# Plant → capacity
plant_capacity = dict(
    zip(
        wh_capacities["Plant ID"],
        wh_capacities["Daily Capacity "]
    )
)

# Plant → warehouse cost per unit
plant_cost = dict(
    zip(
        wh_costs["WH"],
        wh_costs["Cost/unit"]
    )
)

print("Lookup dictionaries created.")
print("Products:", len(product_to_plants))
print("Plants with capacity:", len(plant_capacity))
print("Plants with warehouse costs:", len(plant_cost))
print("Plants with port connections:", len(plant_to_ports))

Lookup dictionaries created.
Products: 1540
Plants with capacity: 19
Plants with warehouse costs: 19
Plants with port connections: 19


## 3. Feasible Option Generation

For each order, we generate the feasible plant and transportation options
that satisfy the basic supply-chain constraints.

A feasible option must satisfy:

- The product is available at the selected plant.
- The selected plant is connected to the order's destination port.
- The carrier and service combination is compatible with the freight data.
- The shipment weight falls within the applicable freight-rate range.
- VMI customer restrictions are respected.

For each feasible option, we calculate:

- Warehouse cost
- Freight cost
- Total cost

The resulting option table will be used by both the Simulated Annealing
and MILP implementations.

## 3.2 VMI Customer Restrictions

Vendor Managed Inventory (VMI) customers have specific plant
requirements in the dataset.

If a customer is listed for a particular plant, that relationship
must be respected when generating feasible options.

We create a lookup dictionary so that the restriction can be checked
efficiently for each order.

In [ ]:
# Create Customer → Allowed Plant lookup

vmi_customer_to_plants = {}

for _, row in vmi_customers.iterrows():

    plant = row["Plant Code"]
    customers = str(row["Customers"]).split(",")

    for customer in customers:

        customer = customer.strip()

        if customer not in vmi_customer_to_plants:
            vmi_customer_to_plants[customer] = []

        vmi_customer_to_plants[customer].append(plant)

print("VMI customers:", len(vmi_customer_to_plants))

VMI customers: 10


## 3.3 Generate Feasible Options

For each order, we identify all feasible plant and transportation
options.

The feasibility checks are:

1. The product must be available at the plant.
2. The plant must be connected to the order's origin port.
3. VMI restrictions must be satisfied when applicable.
4. A valid freight rate must exist for normal freight services.
5. The V44_3 + CRF combination is treated as the special historical
   freight case described in the dataset.

For every feasible option, warehouse cost and transportation cost are
calculated.

    Total Cost = Warehouse Cost + Freight Cost

The resulting table is the common decision-option set used by the
Simulated Annealing and MILP models.

In [ ]:
# Generate feasible options for all cleaned orders
# These options will be used by both SA and MILP.

feasible_options = []

# ---------------------------------------------------------
# 1. Create Product -> Plants mapping
# ---------------------------------------------------------

product_to_plants = (
    products_plant
    .groupby("Product ID")["Plant Code"]
    .apply(list)
    .to_dict()
)


# ---------------------------------------------------------
# 2. Create Plant -> Ports mapping
# ---------------------------------------------------------

plant_to_ports = (
    plant_ports
    .groupby("Plant Code")["Port"]
    .apply(list)
    .to_dict()
)


# ---------------------------------------------------------
# 3. Create Plant -> Warehouse Cost mapping
# ---------------------------------------------------------

plant_cost = dict(
    zip(
        wh_costs["WH"],
        wh_costs["Cost/unit"]
    )
)


# ---------------------------------------------------------
# 4. Create VMI Customer -> Allowed Plants mapping
# ---------------------------------------------------------

vmi_customer_to_plants = {}

for _, row in vmi_customers.iterrows():

    plant = row["Plant Code"]

    # Some rows may contain multiple customers
    customers = str(row["Customers"]).split(",")

    for customer in customers:

        customer = customer.strip()

        if customer:
            vmi_customer_to_plants.setdefault(
                customer, []
            ).append(plant)


# ---------------------------------------------------------
# 5. Generate feasible options for each order
# ---------------------------------------------------------

for _, order in orders.iterrows():

    # Basic order information
    order_id = order["Order ID"]
    product = order["Product ID"]
    customer = order["Customer"]

    origin_port = order["Origin Port"]
    destination_port = order["Destination Port"]

    carrier = order["Carrier"]
    service = order["Service Level"]

    quantity = order["Unit quantity"]
    weight = order["Weight"]


    # -----------------------------------------------------
    # Find plants that can produce the ordered product
    # -----------------------------------------------------

    candidate_plants = product_to_plants.get(
        product, []
    )


    # -----------------------------------------------------
    # Find a valid freight option
    # -----------------------------------------------------
    # V44_3 + CRF is a special case with zero freight cost.

    if carrier == "V44_3" and service == "CRF":

        freight_cost = 0.0
        freight_available = True

    else:

        # Find freight records matching the order
        matching_rates = freight_rates[
            (freight_rates["Carrier"] == carrier) &
            (freight_rates["orig_port_cd"] == origin_port) &
            (freight_rates["dest_port_cd"] == destination_port) &
            (freight_rates["svc_cd"] == service) &
            (freight_rates["minm_wgh_qty"] <= weight) &
            (freight_rates["max_wgh_qty"] >= weight)
        ]

        # Check whether a valid freight rate exists
        if matching_rates.empty:

            freight_available = False

        else:

            # Use the first applicable freight rate
            rate = matching_rates.iloc[0]

            # Freight cost is the greater of:
            # minimum cost or weight × rate
            freight_cost = max(
                rate["minimum cost"],
                weight * rate["rate"]
            )

            freight_available = True


    # Skip the order if no freight option exists
    if not freight_available:
        continue


    # -----------------------------------------------------
    # Check each product-compatible plant
    # -----------------------------------------------------

    for plant in candidate_plants:

        # Plant must be connected to the order's origin port
        if origin_port not in plant_to_ports.get(
            plant, []
        ):
            continue


        # -------------------------------------------------
        # Apply VMI customer restriction
        # -------------------------------------------------

        if customer in vmi_customer_to_plants:

            allowed_plants = vmi_customer_to_plants[
                customer
            ]

            if plant not in allowed_plants:
                continue


        # -------------------------------------------------
        # Calculate warehouse cost
        # -------------------------------------------------

        warehouse_cost = (
            quantity * plant_cost[plant]
        )


        # -------------------------------------------------
        # Calculate total logistics cost
        # -------------------------------------------------

        total_cost = (
            warehouse_cost + freight_cost
        )


        # -------------------------------------------------
        # Store the feasible option
        # -------------------------------------------------

        feasible_options.append({

            "Order ID": order_id,

            "Plant": plant,

            "Origin Port": origin_port,

            "Destination Port": destination_port,

            "Carrier": carrier,

            "Service Level": service,

            "Unit Quantity": quantity,

            "Weight": weight,

            "Warehouse Cost": warehouse_cost,

            "Freight Cost": freight_cost,

            "Total Cost": total_cost
        })


# ---------------------------------------------------------
# Convert the list into a DataFrame
# ---------------------------------------------------------

feasible_options = pd.DataFrame(
    feasible_options
)


# ---------------------------------------------------------
# Display summary
# ---------------------------------------------------------

total_orders = len(orders)

orders_with_options = (
    feasible_options["Order ID"].nunique()
)

orders_without_options = (
    total_orders - orders_with_options
)

print("Total orders:", total_orders)
print("Orders with feasible options:", orders_with_options)
print("Orders without options:", orders_without_options)
print("Total feasible options:", len(feasible_options))

Total orders: 6438
Orders with feasible options: 6438
Orders without options: 0
Total feasible options: 7441


In [ ]:
orders_with_options = feasible_options["Order ID"].nunique()

print("Total orders:", len(orders))
print("Orders with feasible options:", orders_with_options)
print("Orders without options:", len(orders) - orders_with_options)

Total orders: 6438
Orders with feasible options: 6438
Orders without options: 0


In [ ]:
# Planning horizon in days
H = 6

# Create a dictionary containing the effective capacity of each plant
effective_capacity = {}

for _, row in wh_capacities.iterrows():

    plant = row["Plant ID"]
    daily_capacity = row["Daily Capacity "]

    effective_capacity[plant] = daily_capacity * H


print("Planning horizon:", H, "days")
print("Total effective capacity:",
      sum(effective_capacity.values()))

Planning horizon: 6 days
Total effective capacity: 34746


In [ ]:
# Check final plant loads

final_loads = {}

for plant in effective_capacity:
    final_loads[plant] = 0

for order_id, option_index in best_solution.items():
    option = options_by_order[order_id].iloc[option_index]
    plant = option["Plant"]
    final_loads[plant] += 1

# Check capacity violations
capacity_violations = {}

for plant in final_loads:
    if final_loads[plant] > effective_capacity[plant]:
        capacity_violations[plant] = (
            final_loads[plant],
            effective_capacity[plant]
        )

print("Total orders:", len(best_solution))
print("Capacity violations:", len(capacity_violations))

if len(capacity_violations) == 0:
    print("Solution is feasible.")
else:
    print("Capacity violations found:")
    print(capacity_violations)

Total orders: 6438
Capacity violations: 0
Solution is feasible.
